# Pluffy AI server (Colab)
1. **Runtime → Change runtime type → T4 GPU → Save**
2. Run the cell below (play button). It takes about 3 to 6 minutes the first time.
3. Copy the `node pluffy-ai.js ...` command it prints and run it on your PC, in the Pluffy repo folder.
4. Keep this cell running while you generate. When you're finished: **Runtime → Disconnect and delete runtime**.


In [ ]:
import glob, os, re, shutil, subprocess, time, json, urllib.request
from google.colab import drive
drive.mount('/content/drive')

# ---- 1. find the GGUF on Drive and copy it to the fast local disk ----
found = glob.glob('/content/drive/MyDrive/Pluffy-v11-GGUF/*.gguf') or \
        glob.glob('/content/drive/MyDrive/**/*Q4_K_M.gguf', recursive=True)
assert found, 'GGUF not found in Drive. Check the folder name (expected MyDrive/Pluffy-v11-GGUF).'
src = found[0]
print('GGUF:', src, f'({os.path.getsize(src)/1024**3:.2f} GB)')
if not os.path.exists('/content/pluffy.gguf'):
    shutil.copy(src, '/content/pluffy.gguf')

# ---- 2. install Ollama (Colab's GPU is used automatically) ----
assert subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0, 'No GPU. Runtime > Change runtime type > T4 GPU'
if not shutil.which('ollama'):
    subprocess.run('apt-get -qq install -y zstd > /dev/null 2>&1', shell=True)
    r = subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, capture_output=True, text=True)
    assert shutil.which('ollama'), 'Ollama install failed:\n' + (r.stderr or r.stdout)[-1500:]

env = dict(os.environ, OLLAMA_HOST='127.0.0.1:11434', OLLAMA_KEEP_ALIVE='24h')
subprocess.Popen(['ollama', 'serve'], env=env, stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434', timeout=2); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('Ollama did not start. See /content/ollama.log')

# ---- 3. register your model with the training system prompt and settings ----
open('/content/Modelfile', 'w').write("""FROM /content/pluffy.gguf

TEMPLATE \"\"\"<|im_start|>system
{{ .System }}<|im_end|>
<|im_start|>user
{{ .Prompt }}<|im_end|>
<|im_start|>assistant
\"\"\"

SYSTEM \"\"\"You generate one valid Pluff @arkzen DSL block. Output only the requested block, including its opening and closing markers where required.\"\"\"

PARAMETER temperature 0
PARAMETER num_ctx 8192
PARAMETER num_predict 4096
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|im_start|>"
PARAMETER stop "<|endoftext|>"
""")
r = subprocess.run(['ollama', 'create', 'pluffy-v11', '-f', '/content/Modelfile'], env=env, capture_output=True, text=True)
assert r.returncode == 0, 'ollama create failed:\n' + (r.stderr or r.stdout)[-1500:]

# ---- 4. quick self-test (should look like a /* @arkzen:meta ... */ block) ----
t0 = time.time()
req = urllib.request.Request('http://127.0.0.1:11434/api/chat', data=json.dumps({
    'model': 'pluffy-v11', 'stream': False,
    'messages': [{'role': 'user', 'content': 'Write the @arkzen:meta block for this application: a todo app'}],
}).encode(), headers={'Content-Type': 'application/json'})
out = json.loads(urllib.request.urlopen(req, timeout=300).read())
print('\nSELF-TEST OUTPUT:\n' + out['message']['content'])
print(f"\n({out.get('eval_count', '?')} tokens in {time.time()-t0:.1f}s)")

# ---- 5. public tunnel (no account needed) ----
if not os.path.exists('/content/cloudflared'):
    subprocess.run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared', shell=True)
tun = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:11434', '--http-host-header', 'localhost:11434'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
t0 = time.time()
while time.time() - t0 < 60 and not url:
    line = tun.stdout.readline()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line or '')
    if m: url = m.group(0)
assert url, 'Tunnel did not start'
print('\n' + '=' * 70)
print('SERVER READY. On your PC, in the Pluffy repo folder, run:\n')
print(f'node pluffy-ai.js "a todo app" --tables "todos(title:string,done:boolean)" --server {url} --model pluffy-v11')
print('=' * 70)
print('Keep this cell running. Stop it (square button) and use Runtime > Disconnect and delete runtime when done.')
while True:
    time.sleep(60)